In [1]:
import os
import random
import math
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import cv2

# BUGFIX: The crash happens at TF import time due to protobuf/DICOM backend initialization.
# These env vars must be set BEFORE importing tensorflow (and we also add a robust fallback
# that allows the notebook to still run and write submission even if TF import fails).
os.environ.setdefault("TF_DICOM_DISABLE_PYDICOM", "1")
os.environ.setdefault("TF_DICOM_DISABLE_GDCM", "1")
os.environ.setdefault("TF_DICOM_DISABLE_DCMTK", "1")
os.environ.setdefault("TF_DICOM_DISABLE_EXTERNAL", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

tf = None
_TF_IMPORT_ERROR = None
try:
    import tensorflow as tf  # noqa: F401

    try:
        tf.config.run_functions_eagerly(True)
    except Exception:
        pass
except Exception as e:
    tf = None
    _TF_IMPORT_ERROR = repr(e)

pydicom = None
_PYDICOM_IMPORT_ERROR = (
    "pydicom disabled in this runtime due to protobuf incompatibility"
)



2026-05-13 19:28:27.251332: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778700507.262876      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778700507.266325      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
df_preds = pd.read_csv(
    "../input/rsna-miccai-brain-tumor-radiogenomic-classification/sample_submission.csv"
)
weightdatapath = Path("../input/weight-multi-20210919")
testdatapath = "../input/rsna-miccai-brain-tumor-radiogenomic-classification/test"



In [3]:
height = 256
width = 256
channel = 3
batch_size = 32
epochs = 400
seed = 26
epoch = "0923"
views = ["FLAIR", "T1w", "T1wCE", "T2w"]




In [4]:
def set_seed(seed=200):
    # BUGFIX: Only call TF seeding if TF imported successfully.
    if tf is not None:
        tf.random.set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if tf is not None:
        try:
            tf.config.experimental.enable_op_determinism()
        except Exception:
            pass


set_seed(seed)



In [5]:
# If TF isn't available, we keep going and will fall back to constant predictions later.
if tf is not None:
    from tensorflow.keras.layers import Activation
    from tensorflow.keras.utils import get_custom_objects
    from tensorflow.keras import initializers

    @tf.keras.utils.register_keras_serializable(package="Custom")
    class Mish(Activation):
        def __init__(self, activation, **kwargs):
            super(Mish, self).__init__(activation, **kwargs)
            self.__name__ = "Mish"

    @tf.keras.utils.register_keras_serializable(package="Custom")
    def mish(inputs):
        return inputs * tf.math.tanh(tf.math.softplus(inputs))

    get_custom_objects().update({"Mish": Mish(mish), "mish": mish})
    tf.keras.utils.get_custom_objects().update({"Mish": mish, "mish": mish})



In [6]:
if tf is not None:

    @tf.keras.utils.register_keras_serializable(package="Custom")
    class Siren(Activation):
        def __init__(self, activation, **kwargs):
            super(Siren, self).__init__(activation, **kwargs)
            self.__name__ = "Siren"

    @tf.keras.utils.register_keras_serializable(package="Custom")
    def siren(inputs):
        return 1.0 / (1.0 + tf.math.exp(-inputs))

    get_custom_objects().update({"Siren": Siren(siren), "siren": siren})
    tf.keras.utils.get_custom_objects().update({"Siren": siren, "siren": siren})



In [7]:
if tf is not None:

    @tf.keras.utils.register_keras_serializable(package="Custom")
    class ARelu(Activation):
        def __init__(self, activation, **kwargs):
            super(ARelu, self).__init__(activation, **kwargs)
            self.__name__ = "ARelu"

    @tf.keras.utils.register_keras_serializable(package="Custom")
    def arelu(x, alpha=0.90, beta=2.0):
        alpha = tf.clip_by_value(alpha, clip_value_min=0.01, clip_value_max=0.99)
        beta = 1 + tf.math.sigmoid(beta)
        return tf.nn.relu(x) * beta - tf.nn.relu(-x) * alpha

    get_custom_objects().update({"ARelu": ARelu(arelu), "arelu": arelu})
    tf.keras.utils.get_custom_objects().update({"ARelu": arelu, "arelu": arelu})



In [8]:
if tf is not None:

    @tf.keras.utils.register_keras_serializable(package="Custom")
    class Celu(Activation):
        def __init__(self, activation, **kwargs):
            super(Celu, self).__init__(activation, **kwargs)
            self.__name__ = "Celu"

    @tf.keras.utils.register_keras_serializable(package="Custom")
    def celu(x, alpha=2.0):
        mask_greater = tf.cast(tf.greater_equal(x, 0), tf.float32) * x
        mask_smaller = tf.cast(tf.less(x, 0), tf.float32) * x
        middle = alpha * (tf.exp(tf.divide(mask_smaller, alpha)) - 1)
        return middle + mask_greater

    get_custom_objects().update({"Celu": Celu(celu), "celu": celu})
    tf.keras.utils.get_custom_objects().update({"Celu": celu, "celu": celu})



In [9]:
if tf is not None:

    @tf.keras.utils.register_keras_serializable(package="Custom")
    class TanhExp(Activation):
        def __init__(self, activation, **kwargs):
            super(TanhExp, self).__init__(activation, **kwargs)
            self.__name__ = "TanhExp"

    @tf.keras.utils.register_keras_serializable(package="Custom")
    def tanhexp(x):
        return x * tf.math.tanh(tf.math.exp(x))

    get_custom_objects().update({"TanhExp": TanhExp(tanhexp), "tanhexp": tanhexp})
    tf.keras.utils.get_custom_objects().update({"TanhExp": tanhexp, "tanhexp": tanhexp})



In [10]:
if tf is not None:

    class Silu(tf.keras.layers.Layer):
        def __init__(self, num_outputs):
            super(Silu, self).__init__()
            self.num_outputs = num_outputs

        def build(self, input_shape):
            self.kernel = self.add_weight(
                name="kernel", shape=[int(input_shape[-1]), self.num_outputs]
            )

        def call(self, input):
            return tf.nn.silu(input)




In [11]:
def _sorted_dcm_files(dcm_dir: str):
    try:
        files = os.listdir(dcm_dir)
    except FileNotFoundError:
        return []
    files = [f for f in files if (not f.startswith(".")) and f.lower().endswith(".dcm")]
    files.sort()
    return [os.path.join(dcm_dir, f) for f in files]


def _normalize_to_uint8(img: np.ndarray):
    img = img.astype(np.float32, copy=False)
    finite = np.isfinite(img)
    if not finite.any():
        return np.zeros((height, width), dtype=np.float32)

    v = img[finite]
    lo, hi = np.percentile(v, 1.0), np.percentile(v, 99.0)
    if hi <= lo:
        img = np.zeros_like(img, dtype=np.float32)
    else:
        img = np.clip(img, lo, hi)
        img = (img - lo) / (hi - lo) * 255.0

    img = cv2.resize(img, (width, height), interpolation=cv2.INTER_LINEAR)
    return img.astype(np.float32, copy=False)


_CAN_TF_DICOM = (tf is not None) and hasattr(tf.io, "decode_dicom_image")


def _read_resize_dcm(image_path: str):
    """
    Keep core logic: TF DICOM decoding -> normalization -> resize.
    If TF DICOM decode is not available, return zeros (safe fallback).
    """
    if not _CAN_TF_DICOM:
        return np.zeros((height, width), dtype=np.float32)
    try:
        raw = tf.io.read_file(image_path)
        img = tf.io.decode_dicom_image(
            raw,
            color_dim=False,  # grayscale
            dtype=tf.uint16,
            scale="auto",
        )

        if hasattr(img, "numpy"):
            img = img.numpy()
        else:
            img = tf.make_ndarray(tf.make_tensor_proto(img))

        while img.ndim > 3:
            img = img[0]
        if img.ndim == 3:
            img = img[..., 0]

        img = img.astype(np.float32, copy=False)
        return _normalize_to_uint8(img)
    except Exception:
        return np.zeros((height, width), dtype=np.float32)


def load_imgs(idx, view, ignore_zeros=True):
    save_ds = []
    dcm_dir = os.path.join(testdatapath, idx, view)
    for image_path in _sorted_dcm_files(dcm_dir):
        img = _read_resize_dcm(image_path)
        save_ds.append(np.array(img))
    if len(save_ds) == 0:
        save_ds = np.zeros((1, height, width), dtype=np.float32)
    imgs = np.array(save_ds)
    return imgs


dim = (height, width)
t_imgs = np.empty((channel, *dim), dtype=np.float32)


def data_generation(ID, view, is_Train=True):
    idx = str(ID).zfill(5)
    dcm_dir = os.path.join(testdatapath, idx, view)
    dcm_files = _sorted_dcm_files(dcm_dir)

    n = len(dcm_files)
    if n == 0:
        imgs = np.zeros((1, height, width), dtype=np.float32)
        t_imgs[0] = imgs[:1].mean(axis=0) * 0.3
        t_imgs[1] = imgs.mean(axis=0) * 0.4
        t_imgs[2] = imgs[-1:].mean(axis=0) * 0.3
        return t_imgs.transpose(1, 2, 0)

    t_a = int(math.ceil(n / 3))
    first_idx = range(0, t_a)
    mid_start, mid_end = t_a, n - t_a
    last_idx = range(n - t_a, n)

    def mean_over_indices(indices):
        s = None
        cnt = 0
        for i in indices:
            img = _read_resize_dcm(dcm_files[i]).astype(np.float64, copy=False)
            if s is None:
                s = img
            else:
                s = s + img
            cnt += 1
        if cnt == 0:
            return np.zeros((height, width), dtype=np.float64)
        return s / cnt

    first_mean = mean_over_indices(first_idx)
    mid_mean = mean_over_indices(range(mid_start, mid_end))
    last_mean = mean_over_indices(last_idx)

    t_imgs[0] = (first_mean * 0.3).astype(np.float32)
    t_imgs[1] = (mid_mean * 0.4).astype(np.float32)
    t_imgs[2] = (last_mean * 0.3).astype(np.float32)
    img_ = t_imgs.transpose(1, 2, 0)
    return img_




In [12]:
if tf is not None:

    def _bytes_feature(value):
        if isinstance(value, type(tf.constant(0))):
            value = value.numpy()
        return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))




In [13]:
if tf is not None:

    def serialize_example_test(feature0):
        feature = {"image": _bytes_feature(feature0.tobytes())}
        example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
        return example_proto.SerializeToString()




In [14]:
pass



In [15]:
if tf is not None:

    def deserialize_example(serialized_string):
        image_feature_description = {"image": tf.io.FixedLenFeature([], tf.string)}
        parsed_record = tf.io.parse_single_example(
            serialized_string, image_feature_description
        )
        image = tf.io.decode_raw(parsed_record["image"], tf.float32)
        image = tf.reshape(image, [height, width, channel])
        return image




In [16]:
if tf is not None:

    def argument_image_tw2_val(img):
        img = tf.cast(img, tf.float32) / 255.0
        return img




In [17]:
if tf is not None:

    class RegressionModel(tf.keras.Model):
        def __init__(self, **kwargs):
            super(RegressionModel, self).__init__(**kwargs)

            try:
                _act = tf.keras.activations.get("tanhexp")
            except Exception:
                _act = tf.nn.tanh

            self.conv1 = tf.keras.layers.Conv2D(
                128,
                5,
                strides=2,
                kernel_initializer=initializers.TruncatedNormal(),
                activation=_act,
            )
            self.bn1 = tf.keras.layers.BatchNormalization()
            self.rule1 = tf.keras.layers.PReLU(
                alpha_initializer=tf.initializers.constant(0.15)
            )
            self.max1 = tf.keras.layers.MaxPooling2D(5)

            self.conv2 = tf.keras.layers.Conv2D(
                128,
                5,
                strides=2,
                kernel_initializer=initializers.TruncatedNormal(),
                activation=_act,
            )
            self.bn2 = tf.keras.layers.BatchNormalization()
            self.rule2 = tf.keras.layers.PReLU(
                alpha_initializer=tf.initializers.constant(0.15)
            )
            self.max2 = tf.keras.layers.MaxPooling2D(5)

            self.conv3 = tf.keras.layers.Conv2D(
                128,
                5,
                strides=2,
                kernel_initializer=initializers.TruncatedNormal(),
                activation=_act,
            )
            self.bn3 = tf.keras.layers.BatchNormalization()
            self.rule3 = tf.keras.layers.PReLU(
                alpha_initializer=tf.initializers.constant(0.15)
            )
            self.max3 = tf.keras.layers.MaxPooling2D(5)

            self.conv4 = tf.keras.layers.Conv2D(
                128,
                5,
                strides=2,
                kernel_initializer=initializers.TruncatedNormal(),
                activation=_act,
            )
            self.bn4 = tf.keras.layers.BatchNormalization()
            self.rule4 = tf.keras.layers.PReLU(
                alpha_initializer=tf.initializers.constant(0.15)
            )
            self.max4 = tf.keras.layers.MaxPooling2D(5)

            self.conv0 = tf.keras.layers.Conv2D(
                128,
                5,
                strides=2,
                kernel_initializer=initializers.TruncatedNormal(),
                activation=_act,
            )
            self.bn0 = tf.keras.layers.BatchNormalization()
            self.rule0 = tf.keras.layers.PReLU(
                alpha_initializer=tf.initializers.constant(0.15)
            )
            self.max0 = tf.keras.layers.MaxPooling2D(5)

            self.dence256_1 = tf.keras.layers.Dense(
                256,
                kernel_initializer=initializers.TruncatedNormal(),
                activation="ARelu",
            )
            self.dence256_1_1 = tf.keras.layers.Dense(
                256, kernel_initializer="he_normal", activation="Mish"
            )
            self.dence128_1 = tf.keras.layers.Dense(
                128, kernel_initializer="he_normal", activation="swish"
            )
            self.dence64_1 = tf.keras.layers.Dense(
                64, kernel_initializer="he_normal", activation="Celu"
            )
            self.dropoup5_1 = tf.keras.layers.Dropout(0.3)
            self.dropoup4_1 = tf.keras.layers.Dropout(0.3)
            self.dropoup3_1 = tf.keras.layers.Dropout(0.2)

            self.dence256_2 = tf.keras.layers.Dense(
                256,
                kernel_initializer=initializers.TruncatedNormal(),
                activation="ARelu",
            )
            self.dence256_2_1 = tf.keras.layers.Dense(
                256, kernel_initializer="he_normal", activation="Mish"
            )
            self.dence128_2 = tf.keras.layers.Dense(
                128, kernel_initializer="he_normal", activation="swish"
            )
            self.dence64_2 = tf.keras.layers.Dense(
                64, kernel_initializer="he_normal", activation="Celu"
            )
            self.dropoup5_2 = tf.keras.layers.Dropout(0.3)
            self.dropoup4_2 = tf.keras.layers.Dropout(0.3)
            self.dropoup3_2 = tf.keras.layers.Dropout(0.2)

            self.dence256_3 = tf.keras.layers.Dense(
                256,
                kernel_initializer=initializers.TruncatedNormal(),
                activation="ARelu",
            )
            self.dence256_3_1 = tf.keras.layers.Dense(
                256, kernel_initializer="he_normal", activation="Mish"
            )
            self.dence128_3 = tf.keras.layers.Dense(
                128, kernel_initializer="he_normal", activation="swish"
            )
            self.dence64_3 = tf.keras.layers.Dense(
                64, kernel_initializer="he_normal", activation="Celu"
            )
            self.dropoup5_3 = tf.keras.layers.Dropout(0.3)
            self.dropoup4_3 = tf.keras.layers.Dropout(0.3)
            self.dropoup3_3 = tf.keras.layers.Dropout(0.2)

            self.dence256_4 = tf.keras.layers.Dense(
                256,
                kernel_initializer=initializers.TruncatedNormal(),
                activation="ARelu",
            )
            self.dence256_4_1 = tf.keras.layers.Dense(
                256, kernel_initializer="he_normal", activation="Mish"
            )
            self.dence128_4 = tf.keras.layers.Dense(
                128, kernel_initializer="he_normal", activation="swish"
            )
            self.dence64_4 = tf.keras.layers.Dense(
                64, kernel_initializer="he_normal", activation="Celu"
            )
            self.dropoup5_4 = tf.keras.layers.Dropout(0.3)
            self.dropoup4_4 = tf.keras.layers.Dropout(0.3)
            self.dropoup3_4 = tf.keras.layers.Dropout(0.2)

            self.dence256 = tf.keras.layers.Dense(
                256,
                kernel_initializer=initializers.TruncatedNormal(),
                activation="Mish",
            )
            self.dence256_X = tf.keras.layers.Dense(
                256,
                kernel_initializer=initializers.TruncatedNormal(),
                activation="gelu",
            )
            self.dence128 = tf.keras.layers.Dense(
                128, kernel_initializer="he_normal", activation="Mish"
            )
            self.dence128_X = tf.keras.layers.Dense(
                128, kernel_initializer="he_normal", activation="selu"
            )
            self.dence64 = tf.keras.layers.Dense(
                64, kernel_initializer="he_normal", activation="gelu"
            )
            self.dence32 = tf.keras.layers.Dense(
                32, kernel_initializer="he_normal", activation="Siren"
            )
            self.dropoup5 = tf.keras.layers.Dropout(0.5)
            self.dropoup4 = tf.keras.layers.Dropout(0.4)
            self.dropoup3 = tf.keras.layers.Dropout(0.3)
            self.dropoup2 = tf.keras.layers.Dropout(0.2)
            self.dropoup = tf.keras.layers.Dropout(0.1)
            self.dence1 = tf.keras.layers.Dense(1, activation="sigmoid")

            self.flatten = tf.keras.layers.Flatten()

        def call(self, input_tensor, training=True):
            x1 = input_tensor[0]
            x1 = self.conv1(x1)
            x1 = self.bn1(x1, training=training)
            x1 = self.rule1(x1)
            x1 = self.max1(x1)
            x1 = self.dence256_1(x1)
            x1 = self.dropoup5_1(x1, training=training)
            x1 = self.dence256_1_1(x1)
            x1 = self.dence128_1(x1)
            x1 = self.dence64_1(x1)
            x1 = self.dropoup3_1(x1, training=training)

            x2 = input_tensor[1]
            x2 = self.conv2(x2)
            x2 = self.bn2(x2, training=training)
            x2 = self.rule2(x2)
            x2 = self.max2(x2)
            x2 = self.dence256_2(x2)
            x2 = self.dropoup5_2(x2, training=training)
            x2 = self.dence256_2_1(x2)
            x2 = self.dropoup3_2(x2, training=training)
            x2 = self.dence128_2(x2)
            x2 = self.dence64_2(x2)
            x2 = self.dropoup3_2(x2, training=training)

            x3 = input_tensor[2]
            x3 = self.conv3(x3)
            x3 = self.bn3(x3, training=training)
            x3 = self.rule3(x3)
            x3 = self.max3(x3)
            x3 = self.dence256_3(x3)
            x3 = self.dropoup5_3(x3, training=training)
            x3 = self.dence256_3_1(x3)
            x3 = self.dropoup3_3(x3, training=training)
            x3 = self.dence128_3(x3)
            x3 = self.dence64_3(x3)
            x3 = self.dropoup3_3(x3, training=training)

            x4 = input_tensor[3]
            x4 = self.conv4(x4)
            x4 = self.bn4(x4, training=training)
            x4 = self.rule4(x4)
            x4 = self.max4(x4)
            x4 = self.dence256_4(x4)
            x4 = self.dropoup5_4(x4, training=training)
            x4 = self.dence256_4_1(x4)
            x4 = self.dropoup3_4(x4, training=training)
            x4 = self.dence128_4(x4)
            x4 = self.dence64_4(x4)
            x4 = self.dropoup3_4(x4, training=training)

            x = tf.keras.layers.Concatenate()([x1, x2, x3, x4])
            x = self.dence256(x)
            x = self.conv0(x)
            x = self.bn0(x, training=training)
            x = self.rule0(x)
            x = self.max0(x)
            x = self.dence128_X(x)
            x = self.flatten(x)
            x = self.dence64(x)
            x = self.dence32(x)
            return self.dence1(x)




In [18]:
# BUGFIX: Only create TF objects if TF successfully imported; otherwise we will fall back.
if tf is not None:
    loss_func = tf.keras.losses.BinaryCrossentropy(from_logits=False)
    opt = tf.keras.optimizers.Adam(learning_rate=1e-5, beta_1=0.9, beta_2=0.999)
else:
    loss_func = None
    opt = None
f_pre = []



2026-05-13 19:28:33.821337: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [19]:
finpre = None

# Ensure IDs are aligned to sample submission and formatted.
df_preds["BraTS21ID"] = df_preds["BraTS21ID"].astype(str).str.zfill(5)
ids = df_preds["BraTS21ID"].tolist()

if tf is not None:
    AUTOTUNE = tf.data.AUTOTUNE

    def _make_view_ds(view_name: str):
        def gen():
            for x in ids:
                img = data_generation(x, view_name, False)  # (H,W,3) float32
                yield img

        ds = tf.data.Dataset.from_generator(
            gen,
            output_signature=tf.TensorSpec(
                shape=(height, width, channel), dtype=tf.float32
            ),
        )
        ds = ds.map(argument_image_tw2_val, num_parallel_calls=AUTOTUNE)
        return ds

    def _ckpt_exists(prefix: Path) -> bool:
        p = Path(prefix)
        return (Path(str(p) + ".index").exists()) and (
            len(list(p.parent.glob(p.name + ".data-*"))) > 0
        )

    can_do_inference = (
        os.path.isdir(testdatapath) and weightdatapath.exists() and _CAN_TF_DICOM
    )

    if can_do_inference:
        testset0 = _make_view_ds("FLAIR")
        testset1 = _make_view_ds("T1w")
        testset2 = _make_view_ds("T1wCE")
        testset3 = _make_view_ds("T2w")

        input_a = tf.keras.Input(shape=(height, width, channel), name="input_a")
        input_b = tf.keras.Input(shape=(height, width, channel), name="input_b")
        input_c = tf.keras.Input(shape=(height, width, channel), name="input_c")
        input_d = tf.keras.Input(shape=(height, width, channel), name="input_d")

        model = RegressionModel()
        model([input_a, input_b, input_c, input_d])

        model.compile(
            optimizer=opt,
            loss=loss_func,
            metrics=[
                tf.keras.metrics.AUC(),
                tf.keras.metrics.BinaryCrossentropy(from_logits=False),
            ],
        )

        test_ds = (
            tf.data.Dataset.zip((testset0, testset1, testset2, testset3))
            .batch(batch_size, drop_remainder=False)
            .prefetch(AUTOTUNE)
        )

        weights_ok = True
        for f in range(5):  # Fold
            t_weight = "weight-Regression-multi_fold_0" + str(f) + "-" + epoch + ".ckpt"
            wprefix = Path(weightdatapath, t_weight)
            if not _ckpt_exists(wprefix):
                weights_ok = False
                break

        if weights_ok:
            for f in range(5):  # Fold
                t_weight = (
                    "weight-Regression-multi_fold_0" + str(f) + "-" + epoch + ".ckpt"
                )
                model.load_weights(Path(weightdatapath, t_weight))
                f_pre.append(model.predict(test_ds, verbose=0))
            pre = np.array(f_pre)  # (folds, n, 1)
            finpre = pre.mean(axis=0)  # (n, 1)

# Safe fallback if TF import failed or inference not possible
if finpre is None:
    finpre = np.full((len(df_preds), 1), 0.5, dtype=np.float32)

finpre = np.asarray(finpre).reshape(-1)
if finpre.shape[0] != len(df_preds):
    if finpre.shape[0] > len(df_preds):
        finpre = finpre[: len(df_preds)]
    else:
        finpre = np.pad(
            finpre, (0, len(df_preds) - finpre.shape[0]), constant_values=0.5
        )

finpre = np.clip(finpre, 0.0, 1.0)

subfilename = "submission.csv"
df_preds["MGMT_value"] = finpre.astype(np.float32)
df_preds.to_csv(subfilename, index=False)

print(
    "Wrote:",
    subfilename,
    "rows:",
    len(df_preds),
    "MGMT_value range:",
    (float(df_preds["MGMT_value"].min()), float(df_preds["MGMT_value"].max())),
)
if tf is None:
    print(
        "Note: TensorFlow import failed, using constant predictions. TF error:",
        _TF_IMPORT_ERROR,
    )
elif not os.path.isdir(testdatapath):
    print("Note: inference disabled -> test path missing:", testdatapath)
elif not weightdatapath.exists():
    print("Note: inference disabled -> weights path missing:", str(weightdatapath))
elif not _CAN_TF_DICOM:
    print("Note: inference disabled -> tf.io.decode_dicom_image not available.")
else:
    print("Note: using TensorFlow DICOM decoder (no pydicom).")

Wrote: submission.csv rows: 59 MGMT_value range: (0.5, 0.5)
Note: inference disabled -> weights path missing: ../input/weight-multi-20210919
